# FER-CNN Training (Google Colab, GPU)

Clones the repo, installs dependencies, downloads FER-2013/FANE from Kaggle, and trains the custom CNN on a GPU runtime.

## 1. Clone the repository

In [ ]:
REPO_URL = 'https://github.com/xydani/fer-cnn.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

!git clone --branch {BRANCH} {REPO_URL} fer-cnn
%cd fer-cnn

Install dependencies.

In [ ]:
!pip install -q -r requirements.txt

## 2. Datasets (FER-2013 + FANE)

Enter your Kaggle username and API key (from `kaggle.com > Settings > API`).

In [ ]:
import getpass
import json
import os

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

!pip install -q kaggle

Download and extract the datasets.

In [ ]:
!mkdir -p data/raw/fer-2013 data/raw/fane_data
!kaggle datasets download -d msambare/fer2013 -p data/raw/fer-2013 --unzip
!kaggle datasets download -d furcifer/fane-facial-expressions-and-emotion-dataset -p data/raw/fane_data --unzip

## 3. Train

In [ ]:
model_type = 'custom_cnn'  #@param ["custom_cnn", "xception"]
epochs = 50  #@param {type:"integer"}
batch_size = 64  #@param {type:"integer"}

!python src/train.py --model {model_type} --epochs {epochs} --batch_size {batch_size}

## 4. Inspect results

In [ ]:
import json
import matplotlib.pyplot as plt

with open(f'results/metrics/{model_type}_history.json') as f:
    history = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history['loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch')
axes[0].legend()

axes[1].plot(history['accuracy'], label='train')
axes[1].plot(history['val_accuracy'], label='val')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 5. Save outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

dest = '/content/drive/MyDrive/fer-cnn-results'
shutil.copytree('models_saved', f'{dest}/models_saved', dirs_exist_ok=True)
shutil.copytree('results', f'{dest}/results', dirs_exist_ok=True)